# 2. Feature Engineering & Pseudo-labeling

Mục tiêu của notebook này:
1. **Sessionization:** Gom nhóm các hành vi của người dùng thành các phiên (session) theo thời gian (30 phút).
2. **Pseudo-labeling:** Tạo nhãn giả (0) cho các sản phẩm người dùng đã xem nhưng không tương tác sâu (không add to cart, không purchase).
3. **Feature Engineering:** Trích xuất các đặc trưng cơ bản.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### Load Data
Tải một tập con dữ liệu để thử nghiệm logic.

In [2]:
data_path = '../data/raw/2019-Oct.csv'
# Đọc trước 500k dòng để thử nghiệm code chạy nhanh chóng
df = pd.read_csv(data_path, nrows=500000)
df['event_time'] = pd.to_datetime(df['event_time'])
# Quan trọng: Sắp xếp dữ liệu theo user_id và thời gian
df = df.sort_values(by=['user_id', 'event_time']).reset_index(drop=True)
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 08:47:35+00:00,view,1003535,2053013555631882655,electronics.smartphone,samsung,460.50,244951053,91769fdf-461b-4e43-9c73-88a07481b75c
1,2019-10-01 08:48:28+00:00,view,1003535,2053013555631882655,electronics.smartphone,samsung,460.50,244951053,91769fdf-461b-4e43-9c73-88a07481b75c
2,2019-10-01 01:32:09+00:00,view,2501614,2053013564003713919,appliances.kitchen.oven,redmond,164.71,306441847,47641f8a-3aba-471a-8d07-014deccec567
3,2019-10-01 02:56:47+00:00,view,2501614,2053013564003713919,appliances.kitchen.oven,redmond,164.71,306441847,47641f8a-3aba-471a-8d07-014deccec567
4,2019-10-01 07:58:28+00:00,view,2501614,2053013564003713919,appliances.kitchen.oven,redmond,164.71,306441847,47641f8a-3aba-471a-8d07-014deccec567


### 1. Sessionization (Chia phiên)
Tạo `custom_session_id` để gom nhóm các hành vi của cùng một user xảy ra cách nhau dưới 30 phút.

In [3]:
def create_sessions(df, threshold_minutes=30):
    # Tính khoảng thời gian chênh lệch giữa các sự kiện liên tiếp của cùng 1 user
    df['time_diff'] = df.groupby('user_id')['event_time'].diff()
    
    # Đánh dấu sự kiện đầu tiên của 1 session mới (time_diff > 30 mins hoặc time_diff là NaT)
    is_new_session = df['time_diff'].dt.total_seconds() > (threshold_minutes * 60)
    is_new_session = is_new_session | df['time_diff'].isna()
    
    # Tạo session_id bằng cách cộng dồn (cumsum) các mốc new_session
    df['custom_session_id'] = is_new_session.cumsum()
    
    return df.drop(columns=['time_diff'])

df = create_sessions(df)
print(f"Total custom sessions created: {df['custom_session_id'].nunique()}")
df[['user_id', 'event_time', 'event_type', 'product_id', 'custom_session_id']].head(15)

Total custom sessions created: 100419


,user_id,event_time,event_type,product_id,custom_session_id
0,244951053,2019-10-01 08:47:35+00:00,view,1003535,1
1,244951053,2019-10-01 08:48:28+00:00,view,1003535,1
2,306441847,2019-10-01 01:32:09+00:00,view,2501614,2
3,306441847,2019-10-01 02:56:47+00:00,view,2501614,3
4,306441847,2019-10-01 07:58:28+00:00,view,2501614,4
5,306441847,2019-10-01 08:26:14+00:00,view,2501614,4
6,321655812,2019-10-01 05:55:19+00:00,view,17200728,5
7,321655812,2019-10-01 05:56:57+00:00,view,17200527,5
8,330585300,2019-10-01 09:46:41+00:00,view,3701084,6
9,330585300,2019-10-01 09:54:51+00:00,view,3701003,6


### 2. Pseudo-labeling
Trong một phiên, nếu sản phẩm được mua (`purchase`) hoặc đưa vào giỏ hàng (`cart`), ta gán nhãn `1`. Nếu chỉ xem (`view`) mà không có hành động nào khác trong phiên đó, gán nhãn `0`.

In [4]:
def apply_pseudo_labels(df):
    # Group by session_id and product_id to aggregate events
    # Chúng ta dùng 'max' để đơn giản hóa logic:
    # Gán trọng số: view = 0, cart = 1, purchase = 1
    
    # Tạo cột weight tạm thời
    event_weights = {'view': 0, 'cart': 1, 'purchase': 1}
    df['event_weight'] = df['event_type'].map(event_weights)
    
    # Lấy hành động có giá trị cao nhất trong phiên cho mỗi sản phẩm
    labeled_df = df.groupby(['custom_session_id', 'product_id', 'user_id'])['event_weight'].max().reset_index()
    labeled_df.rename(columns={'event_weight': 'label'}, inplace=True)
    
    # Nếu label = 0, nghĩa là negative sample (chỉ view).
    # Nếu label = 1, nghĩa là positive sample (có cart hoặc purchase).
    return labeled_df

labeled_data = apply_pseudo_labels(df)
print("Tỷ lệ phân bố Label:\n", labeled_data['label'].value_counts(normalize=True))
print("\nSố lượng Label:\n", labeled_data['label'].value_counts())
labeled_data.head(10)

Tỷ lệ phân bố Label:
 label
0    0.965845
1    0.034155
Name: proportion, dtype: float64

Số lượng Label:
 label
0    312275
1     11043
Name: count, dtype: int64


,custom_session_id,product_id,user_id,label
0,1,1003535,244951053,0
1,2,2501614,306441847,0
2,3,2501614,306441847,0
3,4,2501614,306441847,0
4,5,17200527,321655812,0
5,5,17200728,321655812,0
6,6,3700006,330585300,0
7,6,3700298,330585300,0
8,6,3700994,330585300,0
9,6,3701003,330585300,0


### 3. Feature Extraction (Trích xuất đặc trưng)
Tạo các đặc trưng cho User (Ví dụ: Tổng số lượt tương tác) và cho Item (Ví dụ: Tổng số lượt xem, Giá trung bình).

In [5]:
def extract_user_features(df):
    # Tính tổng số lượt tương tác của mỗi user
    user_features = df.groupby('user_id').size().reset_index(name='user_total_interactions')
    
    # Tính số lượng phiên duy nhất của mỗi user
    user_sessions = df.groupby('user_id')['custom_session_id'].nunique().reset_index(name='user_total_sessions')
    user_features = user_features.merge(user_sessions, on='user_id', how='left')
    
    return user_features

user_features = extract_user_features(df)
print("User Features:")
user_features.head()

User Features:


,user_id,user_total_interactions,user_total_sessions
0,244951053,2,1
1,306441847,4,3
2,321655812,2,1
3,330585300,6,1
4,332550649,7,1


In [6]:
def extract_item_features(df):
    # Tính tổng số lượt tương tác (view, cart, purchase) của mỗi item
    item_features = df.groupby('product_id').size().reset_index(name='item_total_interactions')
    
    # Tính số lượng user duy nhất đã tương tác với item
    item_users = df.groupby('product_id')['user_id'].nunique().reset_index(name='item_unique_users')
    item_features = item_features.merge(item_users, on='product_id', how='left')
    
    # Lấy giá trị trung bình (nếu có giá trị thay đổi) hoặc giá trị hiện tại
    item_price = df.groupby('product_id')['price'].mean().reset_index(name='item_avg_price')
    item_features = item_features.merge(item_price, on='product_id', how='left')
    
    return item_features

item_features = extract_item_features(df)
print("Item Features:")
item_features.head()

Item Features:


,product_id,item_total_interactions,item_unique_users,item_avg_price
0,1001588,30,17,128.31
1,1002042,7,3,89.81
2,1002062,8,4,77.14
3,1002098,63,46,409.02
4,1002099,355,234,370.41
